### useful links:

https://www.reddit.com/r/learnmachinelearning/comments/gn3bqz/explaining_randomsearchcv_in_sklearn_and_how_to/

# Home Safety Risk Ranking — Simplified Learner Recipe

This notebook guides you through building a **risk-ranking model** for the home-safety dataset.

The goal is **not** to create a perfect yes/no classifier. The goal is to sort records from highest risk to lowest risk so that the top records contain more true incidents than we would get by random selection.

This is important because the target is imbalanced. When positives are rare, accuracy and a default 0.5 threshold can be misleading. Here we care about questions like:

- If we inspect the **top 50** highest-risk records, how many true incidents do we capture?
- Is the top 50 list better than random selection?
- Would top 100 or top 200 be more realistic operationally?

This simplified version focuses on **one model family: Random Forest**.

## Before you start

You only need to make a small number of decisions:

1. Update the data file path.
2. Confirm the target column name.
3. Check the target is correctly encoded as 0/1.
4. Review the list of columns removed from modelling to avoid leakage.
5. Run the RandomizedSearchCV section and interpret the top-K results.

Avoid changing too many things at once. Get the notebook running first, then improve it.

## 1. Imports

Run this cell first. It loads the packages needed for the workflow.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import randint

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Simple configuration

Update only the values in this cell.

The most important settings are:

- `DATA_PATH`: where the dataset is saved.
- `TARGET_RAW`: the original target column in the dataset.
- `TOP_K_VALUES`: the ranked list sizes to evaluate.

In [2]:
# TODO 1: update this path to match where your dataset is saved.
#DATA_PATH = Path("df_new.xlsx")
#DATA_PATH = Path("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/PYTHON_OUTPUTS/df_new.xlsx")

##df_new_2 HAS ALL THE POST INCIDENT DATA REMOVED

DATA_PATH = Path("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/PYTHON_OUTPUTS/df_new_2.xlsx")
# TODO 2: confirm this is the correct target column in your dataset.
TARGET_RAW = "Incident?"


# This will be the cleaned 0/1 target column used for modelling.
TARGET = "target_incident"

# These are the ranked list sizes we want to evaluate.
# Keep this simple at first.
TOP_K_VALUES = [50, 100, 200, 500]

# Random seed so results are reproducible.
RANDOM_STATE = 42

# Data split sizes.
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN = 0.25

# Random search size.
# Start with 25. Increase later if the notebook runs quickly.
N_ITER_SEARCH = 1

# Cross-validation folds.
CV_SPLITS = 2

In [3]:
DATA_PATH

WindowsPath('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/PYTHON_OUTPUTS/df_new_2.xlsx')

## 3. Load the data

This cell tries to load Excel, CSV, or Parquet files.

After loading, check:

- the number of rows and columns;
- whether the target column exists;
- whether the first few rows look sensible.

In [4]:
def load_tabular_data(path: Path) -> pd.DataFrame:
    """Load a tabular dataset from Excel, CSV, or Parquet."""
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {suffix}")


df_raw = load_tabular_data(DATA_PATH)

print("Dataset shape:", df_raw.shape)
display(df_raw.head(3))

print("\nColumns:")
print(df_raw.columns.tolist())

Dataset shape: (82314, 25)


,Unnamed: 0,Addressbase UPRN,ABP_Classification_Desc,Tenure_Desc,AgeBracket_EldParent,(H) Age - Fine,(H) Presence of Elderly Parent,(H) Number of Adults in Household,(H) Family Lifestage v3,(H) Length of Residency,...,Household Acorn Category Description,Household Acorn Type Description_x,Town,Station_Ground_Code,LSOA11CD,Local Custodian Name,Household Acorn Group Description,Easting,Northing,Incident?
0,0,10002188486,Detached,Owner,76+,11,0,4,12,11,...,Affluent Achievers,Flourishing families,LECHLADE,JX12,E01028762,West Oxfordshire,Prestigious Properties,424567.0,199327.0,N
1,1,10002188492,Semi-Detached,Owner,61 - 65,8,0,2,9,11,...,Affluent Achievers,Middle-aged suburbanites,LECHLADE,JX12,E01028762,West Oxfordshire,Wealthy Residences,424849.0,199344.0,N
2,2,10002188491,Semi-Detached,Owner,56 - 60,7,0,3,8,11,...,Affluent Achievers,Accomplished suburban families,LECHLADE,JX12,E01028762,West Oxfordshire,Wealthy Residences,424844.0,199344.0,N



Columns:
['Unnamed: 0', 'Addressbase UPRN', 'ABP_Classification_Desc', 'Tenure_Desc', 'AgeBracket_EldParent', '(H) Age - Fine', '(H) Presence of Elderly Parent', '(H) Number of Adults in Household', '(H) Family Lifestage v3', '(H) Length of Residency', '(H) Affluence v2', '(H) Water Poverty Flag', '(H) Fuel Poverty v2 Flag', '(H) Number of Children v3', '(H) Household Income v3 - Bands', 'Household Acorn Category Description', 'Household Acorn Type Description_x', 'Town', 'Station_Ground_Code', 'LSOA11CD', 'Local Custodian Name', 'Household Acorn Group Description', 'Easting', 'Northing', 'Incident?']


In [5]:
'''
df_raw.drop(['Unnamed: 0','ABP_Classification_Code','FRSIncidentIdentifier', 'Year', 'Fiscal_Year',
            'TimeOfCall', 'ResponsiblePartyStationId', 'IncidentCategory',
            'Property_Type', 'Property_Description', 'Description',
            'Property_Code', 'VictimsInvolved', 'VictimType',
            'WasRescued', 'Victim_Category',
            'EvacuationAssistanceInvolved', 'EquipmentUsed', 'CREATION_DATE', 'Household Acorn Group Description',
            'Household Acorn Type Description_y'
            ], axis=1, inplace =True)
'''
df_raw.sample(5)
df_raw.shape

(82314, 25)

In [6]:
df_raw.drop(['Unnamed: 0','AgeBracket_EldParent'], axis=1, inplace =True)
df_raw.sample(3)

,Addressbase UPRN,ABP_Classification_Desc,Tenure_Desc,(H) Age - Fine,(H) Presence of Elderly Parent,(H) Number of Adults in Household,(H) Family Lifestage v3,(H) Length of Residency,(H) Affluence v2,(H) Water Poverty Flag,...,Household Acorn Category Description,Household Acorn Type Description_x,Town,Station_Ground_Code,LSOA11CD,Local Custodian Name,Household Acorn Group Description,Easting,Northing,Incident?
9858,10011925783,Detached,Owner,9,0,2,12,4,18,0,...,Affluent Achievers,Detached singles,KIDLINGTON,JX06,E01028502,Cherwell,Affluent Addresses,456053.04,215761.45,N
40250,10024177373,Detached,Owner,5,0,2,10,4,16,0,...,Affluent Achievers,Exclusive empty nesters,CARTERTON,JX14,E01028772,West Oxfordshire,Wealthy Residences,427715.56,208477.10,N
49228,10011880133,Detached,Owner,9,0,4,10,9,15,0,...,Affluent Achievers,Detached singles,BICESTER,JX07,E01028462,Cherwell,Wealthy Residences,458162.00,223495.00,N


In [7]:
df_raw.columns
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 82314 entries, 0 to 82313
Data columns (total 23 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Addressbase UPRN                      82314 non-null  int64  
 1   ABP_Classification_Desc               82314 non-null  str    
 2   Tenure_Desc                           82314 non-null  str    
 3   (H) Age - Fine                        82314 non-null  int64  
 4   (H) Presence of Elderly Parent        82314 non-null  int64  
 5   (H) Number of Adults in Household     82314 non-null  int64  
 6   (H) Family Lifestage v3               82314 non-null  int64  
 7   (H) Length of Residency               82314 non-null  int64  
 8   (H) Affluence v2                      82314 non-null  int64  
 9   (H) Water Poverty Flag                82314 non-null  int64  
 10  (H) Fuel Poverty v2 Flag              82314 non-null  str    
 11  (H) Number of Children v3 

In [8]:
df_raw[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag']].head(3)

,(H) Fuel Poverty v2 Flag,(H) Presence of Elderly Parent,(H) Water Poverty Flag
0,N,0,0
1,N,0,0
2,N,0,0


## 4. Clean the <u>TARGET</u>

The model needs the target to be numeric:

- `1` = incident / positive case;
- `0` = no incident / negative case.

Check the printed value counts carefully. If the mapping is wrong, fix it before continuing.

In [9]:
def clean_binary_target(series: pd.Series) -> pd.Series:
    """Convert a Yes/No or 0/1 target into clean integer 0/1 values."""
    # If already numeric 0/1, keep it simple.
    numeric = pd.to_numeric(series, errors="coerce")
    non_missing_numeric = numeric.dropna()
    if len(non_missing_numeric) > 0 and set(non_missing_numeric.unique()).issubset({0, 1}):
        return numeric.astype("Int64")

    # Otherwise use a conservative string mapping.
    normalised = series.astype(str).str.strip().str.lower()

    positive_values = {"yes", "y", "Y", "true", "1", "incident", "fire"}
    negative_values = {"no", "n", "N", "false", "0", "none", "no incident", "not incident"}

    mapped = normalised.map(lambda x: 1 if x in positive_values else 0 if x in negative_values else np.nan)
    return mapped.astype("Int64")


print("Original target values:")
display(df_raw[TARGET_RAW].value_counts(dropna=False))

# Create modelling dataframe.
df = df_raw.copy()
df[TARGET] = clean_binary_target(df[TARGET_RAW])

# Drop rows where the target could not be mapped.
before = len(df)
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)
after = len(df)

print(f"Rows before target cleaning: {before}")
print(f"Rows after target cleaning:  {after}")

print("\nCleaned target values:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

Original target values:


Incident?
N    79779
Y     2535
Name: count, dtype: int64

Rows before target cleaning: 82314
Rows after target cleaning:  82314

Cleaned target values:


,count
target_incident,
0,79779
1,2535


### repetetiton for the (H) Fuel Poverty v2 Flag feature

In [10]:

display(df_raw['(H) Fuel Poverty v2 Flag'].value_counts(dropna=False))
'''
# Create modelling dataframe.
#df = df_raw.copy()
df['(H) Fuel Poverty v2 Flag'] = clean_binary_target(df['(H) Fuel Poverty v2 Flag'])

# Drop rows where the target could not be mapped.
before = len(df)
df = df.dropna(subset=['(H) Fuel Poverty v2 Flag']).copy()
df[TARGET] = df['(H) Fuel Poverty v2 Flag'].astype(int)
after = len(df)

print(f"Rows before target cleaning: {before}")
print(f"Rows after target cleaning:  {after}")

print("\nCleaned target values:")
display(df['(H) Fuel Poverty v2 Flag'].value_counts(dropna=False).to_frame("count"))
'''

(H) Fuel Poverty v2 Flag
N    79433
Y     2881
Name: count, dtype: int64

'\n# Create modelling dataframe.\n#df = df_raw.copy()\ndf[\'(H) Fuel Poverty v2 Flag\'] = clean_binary_target(df[\'(H) Fuel Poverty v2 Flag\'])\n\n# Drop rows where the target could not be mapped.\nbefore = len(df)\ndf = df.dropna(subset=[\'(H) Fuel Poverty v2 Flag\']).copy()\ndf[TARGET] = df[\'(H) Fuel Poverty v2 Flag\'].astype(int)\nafter = len(df)\n\nprint(f"Rows before target cleaning: {before}")\nprint(f"Rows after target cleaning:  {after}")\n\nprint("\nCleaned target values:")\ndisplay(df[\'(H) Fuel Poverty v2 Flag\'].value_counts(dropna=False).to_frame("count"))\n'

In [11]:
df[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag']].head(3)

,(H) Fuel Poverty v2 Flag,(H) Presence of Elderly Parent,(H) Water Poverty Flag
0,N,0,0
1,N,0,0
2,N,0,0


## 5. Check class imbalance

This tells us how rare the positive class is.

If the positive class is rare, a model can have high accuracy while still being useless. That is why this notebook focuses on ranking metrics instead.

In [12]:
target_summary = df[TARGET].value_counts().sort_index().to_frame("count")
target_summary["proportion"] = target_summary["count"] / target_summary["count"].sum()
display(target_summary)

positive_rate = df[TARGET].mean()

print(f"Baseline positive rate: {positive_rate:.2%}")
print("This is the approximate success rate expected from random selection.")

,count,proportion
target_incident,,
0,79779,0.969203
1,2535,0.030797


Baseline positive rate: 3.08%
This is the approximate success rate expected from random selection.


## 6. Choose feature columns and avoid leakage

Some columns should not be used as model features.

Common examples:

- unique identifiers;
- post-incident information;
- columns that directly reveal the target;
- columns only known after the incident has happened;
- raw coordinate columns, unless you can justify their use.

The columns removed from modelling can still be kept later in the ranked output so the organisation can identify the records.

In [13]:
# TODO 3: Review this list. Add/remove columns based on your dataset.
# These columns will NOT be used as model features.
DROP_COLUMNS_AS_FEATURES = [
    TARGET_RAW,
    TARGET,
    "Addressbase UPRN",
    "Unnamed: 0",
    "Easting",
    "Northing",
    "FRSIncidentIdentifier",
    "IncidentCategory",
    "VictimsInvolved",
    "VictimType",
    "WasRescued",
    # added by me
    #'Victim_Category', 'EvacuationAssistanceInvolved', 'EquipmentUsed'
   
]

# These columns are useful for the final ranked output, if they exist.
ID_COLUMNS_FOR_OUTPUT = [
    "Addressbase UPRN",
    "Easting",
    "Northing",
    "LSOA11CD",
    "Property_Type",
    "Property_Description",

    # added by me
    #'Victim_Category', 'EvacuationAssistanceInvolved', 'EquipmentUsed'
]

feature_columns = [
    col for col in df.columns
    if col not in DROP_COLUMNS_AS_FEATURES
]

id_columns = [
    col for col in ID_COLUMNS_FOR_OUTPUT
    if col in df.columns
]

print(f"Number of feature columns: {len(feature_columns)}")
print("\nFeature columns used by the model:")
print(feature_columns)

print("\nID/context columns kept for ranked output:")
print(id_columns)


Number of feature columns: 19

Feature columns used by the model:
['ABP_Classification_Desc', 'Tenure_Desc', '(H) Age - Fine', '(H) Presence of Elderly Parent', '(H) Number of Adults in Household', '(H) Family Lifestage v3', '(H) Length of Residency', '(H) Affluence v2', '(H) Water Poverty Flag', '(H) Fuel Poverty v2 Flag', '(H) Number of Children v3', '(H) Household Income v3 - Bands', 'Household Acorn Category Description', 'Household Acorn Type Description_x', 'Town', 'Station_Ground_Code', 'LSOA11CD', 'Local Custodian Name', 'Household Acorn Group Description']

ID/context columns kept for ranked output:
['Addressbase UPRN', 'Easting', 'Northing', 'LSOA11CD']


## 7. Split the data

We use three sets:

- **Train**: fit the model and tune hyperparameters.
- **Validation**: compare the tuned model and inspect ranking performance.
- **Test**: final honest evaluation, used only once at the end.

Because the target is imbalanced, we use stratified splits so each set has a similar positive rate.

In [14]:
X = df[feature_columns].copy()
X.head(2)

,ABP_Classification_Desc,Tenure_Desc,(H) Age - Fine,(H) Presence of Elderly Parent,(H) Number of Adults in Household,(H) Family Lifestage v3,(H) Length of Residency,(H) Affluence v2,(H) Water Poverty Flag,(H) Fuel Poverty v2 Flag,(H) Number of Children v3,(H) Household Income v3 - Bands,Household Acorn Category Description,Household Acorn Type Description_x,Town,Station_Ground_Code,LSOA11CD,Local Custodian Name,Household Acorn Group Description
0,Detached,Owner,11,0,4,12,11,19,0,N,0,8,Affluent Achievers,Flourishing families,LECHLADE,JX12,E01028762,West Oxfordshire,Prestigious Properties
1,Semi-Detached,Owner,8,0,2,9,11,16,0,N,0,3,Affluent Achievers,Middle-aged suburbanites,LECHLADE,JX12,E01028762,West Oxfordshire,Wealthy Residences


# 29/6/26 :
#### Feedback from Goncalo:
#### Instead, please convert genuine Boolean columns to numeric 0/1 values before splitting the data 
#### into train, validation and test sets.
## Immediately after defining X, and before the train/test split, add:

boolean_features = X.select_dtypes(include=["bool"]).columns.tolist()

for column in boolean_features:
    X[column] = X[column].astype(int)

print("Boolean columns converted to 0/1:")
print(boolean_features)

In [15]:
boolean_features = X.select_dtypes(include=["bool"]).columns.tolist()
boolean_features

[]

In [16]:
for column in boolean_features:
    X[column] = X[column].astype(int)

print("Boolean columns converted to 0/1:")
print(boolean_features)

Boolean columns converted to 0/1:
[]


In [ ]:
#X = df[feature_columns].astype(str)
#X.info()

In [17]:
#X = df[feature_columns].copy()
y = df[TARGET].copy()
ids = df[id_columns].copy() if id_columns else pd.DataFrame(index=df.index)

# First split: train+validation vs test.
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)



# Second split: train vs validation.
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=VALIDATION_SIZE_WITHIN_TRAIN,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)



#### added by me 22/6/26
#X_train.astype('str')
#X_test.astype('str')
#X_valid.astype('str')


split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "positives": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)
print(f"positive_rates should remain consistent across train_validation_test split")

,rows,positives,positive_rate
train,49388,1521,0.030797
validation,16463,507,0.030796
test,16463,507,0.030796


positive_rates should remain consistent across train_validation_test split


## 8. Build the preprocessing pipeline

The preprocessing step handles:

- missing numeric values;
- missing categorical values;
- scaling numeric features;
- one-hot encoding categorical features.

Putting preprocessing inside the pipeline helps avoid data leakage during cross-validation.

In [18]:
#numeric_features = X_train.select_dtypes(include=["bool" ,"int", "number"]).columns.tolist()  #, removed on 23/6/26 afternoon   ### bool" ,
#categorical_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(    include=["object", "category", "string"]).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Compatible with different scikit-learn versions.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

Numeric features: 9
Categorical features: 10


In [19]:
categorical_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation.

In [20]:
a = list(X_train[numeric_features])     # make an assert statement for this part
b = list(X_train[categorical_features])
res = set(a) & set(b)

if res:
    print("Common elements exist:", res)
else:
    print("No common elements exist.")

No common elements exist.


## 9. Helper functions for ranking evaluation

These functions are provided for you.

The most important table is the **top-K table**. It answers:

> If we inspect only the top K highest-scored records, how many true positives do we capture?

Important distinction:

- `precision@K` = true positives in top K / K.
- `recall@K` = true positives in top K / all positives in the dataset.

So top-50 recall can look low even when precision is high, because the denominator is all positives, not 50.

In [21]:
def get_scores(model, X_data):
    """Return the model's positive-class probability scores."""
    return model.predict_proba(X_data)[:, 1]


def top_k_capture_table(y_true, scores, k_values, label="data"):
    """Create a top-K ranking evaluation table."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    n_records = len(y_array)
    total_positives = int(y_array.sum())
    baseline_rate = y_array.mean()

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    rows = []
    for k in k_values:
        k_eff = min(k, n_records)
        positives_in_top_k = int(y_sorted[:k_eff].sum())

        precision_at_k = positives_in_top_k / k_eff if k_eff > 0 else np.nan
        recall_at_k = positives_in_top_k / total_positives if total_positives > 0 else np.nan
        lift_at_k = precision_at_k / baseline_rate if baseline_rate > 0 else np.nan

        # Top K cannot capture more than K positives.
        max_possible_recall = min(k_eff, total_positives) / total_positives if total_positives > 0 else np.nan
        pct_of_max_possible = recall_at_k / max_possible_recall if max_possible_recall > 0 else np.nan

        rows.append({
            "dataset": label,
            "k": k_eff,
            "positives_in_top_k": positives_in_top_k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "max_possible_recall_at_k": max_possible_recall,
            "pct_of_max_possible_recall": pct_of_max_possible,
            "lift_at_k": lift_at_k,
        })

    return pd.DataFrame(rows)


def cumulative_gain_frame(y_true, scores):
    """Return data for a cumulative gains curve."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    cumulative_positives = np.cumsum(y_sorted)
    total_positives = y_sorted.sum()

    return pd.DataFrame({
        "inspected_fraction": np.arange(1, len(y_sorted) + 1) / len(y_sorted),
        "cumulative_capture_rate": cumulative_positives / total_positives if total_positives > 0 else np.nan,
    })

## 10. Fit a simple baseline Random Forest

This gives a starting point before hyperparameter tuning.

Do not worry if performance is not perfect. We are checking whether the model has useful ranking signal.

In [22]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 49388 entries, 82167 to 25698
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   ABP_Classification_Desc               49388 non-null  str   
 1   Tenure_Desc                           49388 non-null  str   
 2   (H) Age - Fine                        49388 non-null  int64 
 3   (H) Presence of Elderly Parent        49388 non-null  int64 
 4   (H) Number of Adults in Household     49388 non-null  int64 
 5   (H) Family Lifestage v3               49388 non-null  int64 
 6   (H) Length of Residency               49388 non-null  int64 
 7   (H) Affluence v2                      49388 non-null  int64 
 8   (H) Water Poverty Flag                49388 non-null  int64 
 9   (H) Fuel Poverty v2 Flag              49388 non-null  str   
 10  (H) Number of Children v3             49388 non-null  int64 
 11  (H) Household Income v3 - Bands       49

#### The problem being that the algorithm does not like mixed data types such as str and object being in the same list.
#### On 23/6/26 this was fixed by 
string_cols = X_train.select_dtypes(include=["string","object"]).columns
X_train[string_cols] = X_train[string_cols].astype("string")
X_train.info()

In [23]:
string_cols = X_train.select_dtypes(include=["string","object"]).columns
X_train[string_cols] = X_train[string_cols].astype("string")
X_train.info()

<class 'pandas.DataFrame'>
Index: 49388 entries, 82167 to 25698
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   ABP_Classification_Desc               49388 non-null  string
 1   Tenure_Desc                           49388 non-null  string
 2   (H) Age - Fine                        49388 non-null  int64 
 3   (H) Presence of Elderly Parent        49388 non-null  int64 
 4   (H) Number of Adults in Household     49388 non-null  int64 
 5   (H) Family Lifestage v3               49388 non-null  int64 
 6   (H) Length of Residency               49388 non-null  int64 
 7   (H) Affluence v2                      49388 non-null  int64 
 8   (H) Water Poverty Flag                49388 non-null  int64 
 9   (H) Fuel Poverty v2 Flag              49388 non-null  string
 10  (H) Number of Children v3             49388 non-null  int64 
 11  (H) Household Income v3 - Bands       49

In [24]:
baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs= 1,
    )),
])


baseline_rf.fit(X_train, y_train)

baseline_valid_scores = get_scores(baseline_rf, X_valid)

print("Baseline Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, baseline_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, baseline_valid_scores):.3f}")

display(top_k_capture_table(y_valid, baseline_valid_scores, TOP_K_VALUES, label="validation"))


Baseline Random Forest — validation
ROC-AUC:           0.749
Average Precision: 0.275


,dataset,k,positives_in_top_k,precision_at_k,recall_at_k,max_possible_recall_at_k,pct_of_max_possible_recall,lift_at_k
0,validation,50,36,0.720,0.071006,0.098619,0.720,23.379408
1,validation,100,52,0.520,0.102564,0.197239,0.520,16.885128
2,validation,200,129,0.645,0.254438,0.394477,0.645,20.944053
3,validation,500,171,0.342,0.337278,0.986193,0.342,11.105219


In [25]:
baseline_rf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 11. Tune the Random Forest with RandomizedSearchCV

This searches a wider set of Random Forest settings than a small manual grid.

We tune using **Average Precision** because the target is imbalanced and we care about ranking positives near the top. We still report ROC-AUC because it tells us whether the model has general ranking signal.

Start with `N_ITER_SEARCH = 25`. If the notebook runs quickly, increase it to 50 or 75.

In [26]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs= 1, ## day time 1 = less CPU
    )),
])

param_distributions = {
    "model__n_estimators": randint(300, 1200),    ## may need thier ranges altered -- based upon best_estimator_
    "model__max_depth": [None, 5, 8, 12, 16, 24, 32],
    "model__min_samples_split": randint(2, 50),
    "model__min_samples_leaf": randint(1, 25),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "model__bootstrap": [True, False],
}

minority_count = int(y_train.value_counts().min())
n_splits = max(2, min(CV_SPLITS, minority_count))

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,
)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
    },
    refit="average_precision",
    cv=cv,
    n_jobs= 1,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=True,
    error_score = "raise"   # added onm 29/6/26
)

In [27]:
import time
start_time = time.time()
#start_time


In [28]:
rf_search.fit(X_train, y_train)

Fitting 2 folds for each of 1 candidates, totalling 2 fits


AttributeError: 'bool' object has no attribute 'all'

AttributeError: 'bool' object has no attribute 'all'

RandomizedSearchCV(cv=StratifiedKFold(n_splits=2, random_state=42, shuffle=True),
                   error_score='raise',
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['(H) '
                                                                                'Age '
                                                                                '- '
          

In [29]:
import sklearn

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print(X_train.dtypes.value_counts())

pandas: 3.0.1
scikit-learn: 1.8.0
string    10
int64      9
Name: count, dtype: int64


In [ ]:
'''try:
    rf_search.fit(X_train, y_train)
except AttributeError:
    print("Attribute Error skimmed over.")'''  ## prevents bool error pop up

In [30]:
print("Best CV Average Precision:", rf_search.best_score_)
print("Best parameters:")
for key, value in rf_search.best_params_.items():
    print(f"  {key}: {value}")

Best CV Average Precision: 0.1324094399967342
Best parameters:
  model__bootstrap: True
  model__class_weight: None
  model__max_depth: 32
  model__max_features: 0.3
  model__min_samples_leaf: 8
  model__min_samples_split: 22
  model__n_estimators: 914


In [31]:
print("pipeline_Random_Forest fitted\n--- %s seconds ---" % (time.time() - start_time))

pipeline_Random_Forest fitted
--- 891.6896235942841 seconds ---


12. Evaluate the tuned Random Forest on validation data
This checks whether tuning improved the model on unseen validation data.

Focus especially on:

ROC-AUC;
Average Precision;
positives captured in the top 50, 100, and 200;
lift compared with random selection;
percentage of the maximum possible recall@K.



In [32]:
best_rf = rf_search.best_estimator_
tuned_valid_scores = get_scores(best_rf, X_valid)

print("Tuned Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, tuned_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, tuned_valid_scores):.3f}")

valid_top_k = top_k_capture_table(y_valid, tuned_valid_scores, TOP_K_VALUES, label="validation")
display(valid_top_k)

Tuned Random Forest — validation
ROC-AUC:           0.737
Average Precision: 0.169


,dataset,k,positives_in_top_k,precision_at_k,recall_at_k,max_possible_recall_at_k,pct_of_max_possible_recall,lift_at_k
0,validation,50,28,0.56,0.055227,0.098619,0.56,18.183984
1,validation,100,41,0.41,0.080868,0.197239,0.41,13.313274
2,validation,200,66,0.33,0.130178,0.394477,0.33,10.715562
3,validation,500,110,0.22,0.216963,0.986193,0.22,7.143708


13. Compare baseline and tuned model
Use this section to decide whether tuning actually helped.

A model is not automatically better just because it was tuned. It should improve the validation results or provide a better operational ranking.

In [33]:
comparison_rows = []

for model_name, scores in [
    ("Baseline RF", baseline_valid_scores),
    ("Tuned RF", tuned_valid_scores),
]:
    top_k = top_k_capture_table(y_valid, scores, TOP_K_VALUES, label="validation")
    top_50_row = top_k[top_k["k"] == min(50, len(y_valid))].iloc[0]

    comparison_rows.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_valid, scores),
        "average_precision": average_precision_score(y_valid, scores),
        "top_50_true_positives": top_50_row["positives_in_top_k"],
        "precision_at_50": top_50_row["precision_at_k"],
        "recall_at_50": top_50_row["recall_at_k"],
        "pct_of_max_possible_recall_at_50": top_50_row["pct_of_max_possible_recall"],
        "lift_at_50": top_50_row["lift_at_k"],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

,model,roc_auc,average_precision,top_50_true_positives,precision_at_50,recall_at_50,pct_of_max_possible_recall_at_50,lift_at_50
0,Baseline RF,0.749241,0.275054,36,0.72,0.071006,0.72,23.379408
1,Tuned RF,0.737432,0.169491,28,0.56,0.055227,0.56,18.183984


In [ ]:
import time
start_time = time.time()

In [ ]:
rf_search.fit(X_train, y_train)

In [ ]:
print("pipeline_Random_Forest fitted\n--- %s seconds ---" % (time.time() - start_time))

In [ ]:
print("Best CV Average Precision:", rf_search.best_score_)
print("Best parameters:")
for key, value in rf_search.best_params_.items():
    print(f"  {key}: {value}")

## 12. Evaluate the tuned Random Forest on validation data

This checks whether tuning improved the model on unseen validation data.

Focus especially on:

- ROC-AUC;
- Average Precision;
- positives captured in the top 50, 100, and 200;
- lift compared with random selection;
- percentage of the maximum possible recall@K.

In [ ]:
best_rf = rf_search.best_estimator_
tuned_valid_scores = get_scores(best_rf, X_valid)

print("Tuned Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, tuned_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, tuned_valid_scores):.3f}")

valid_top_k = top_k_capture_table(y_valid, tuned_valid_scores, TOP_K_VALUES, label="validation")
display(valid_top_k)

## 13. Compare baseline and tuned model

Use this section to decide whether tuning actually helped.

A model is not automatically better just because it was tuned. It should improve the validation results or provide a better operational ranking.

In [ ]:
comparison_rows = []
k = 50
for model_name, scores in [
    ("Baseline RF", baseline_valid_scores),
    ("Tuned RF", tuned_valid_scores),
]:
    top_k = top_k_capture_table(y_valid, scores, TOP_K_VALUES, label="validation")
    top_50_row = top_k[top_k["k"] == min(k, len(y_valid))].iloc[0]

    comparison_rows.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_valid, scores),
        "average_precision": average_precision_score(y_valid, scores),
        "top_50_true_positives": top_50_row["positives_in_top_k"],
        "precision_at_50": top_50_row["precision_at_k"],
        "recall_at_50": top_50_row["recall_at_k"],
        "pct_of_max_possible_recall_at_50": top_50_row["pct_of_max_possible_recall"],
        "lift_at_50": top_50_row["lift_at_k"],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# 1. Prepare your new dataframe
df_new = pd.DataFrame({'Data': [7, 8, 9]})
file_name = pd.read_excel('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/file_name.xlsx')
file_name = file_name


import pandas as pd

# 1. Prepare your dataframes


# 2. Export to new sheets
with pd.ExcelWriter(f'file_name {N_ITER_SEARCH} {CV_SPLITS}', engine='openpyxl') as writer:
    comparison.to_excel(writer, sheet_name=(f'iter{N_ITER_SEARCH} {CV_SPLITS}'), index=False)
    #df2.to_excel(writer, sheet_name='Second Sheet', index=False)


## 14. Plot the validation cumulative gains curve

This plot shows how quickly the model captures true positives as more records are inspected.

A useful model should rise faster than the diagonal/random line.

In [ ]:
gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)

ax = gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — tuned Random Forest",
)
ax.set_xlabel("Fraction of records inspected")
ax.set_ylabel("Fraction of positives captured")
plt.show()

## 15. Final test evaluation

Only run this after you have finished choosing the model using the validation set.

Do not tune the model again after looking at the test results.

In [ ]:
final_model = best_rf

test_scores = get_scores(final_model, X_test)

print("Final tuned Random Forest — test")
print(f"ROC-AUC:           {roc_auc_score(y_test, test_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_test, test_scores):.3f}")

test_top_k = top_k_capture_table(y_test, test_scores, TOP_K_VALUES, label="test")
display(test_top_k)

## 16. Create a ranked test output

This creates a table sorted from highest predicted risk to lowest predicted risk.

The top rows are the records the organisation would prioritise for inspection or intervention.

In [ ]:
ranked_test = ids_test.copy()
ranked_test["true_target"] = y_test.values
ranked_test["risk_score"] = test_scores
ranked_test["risk_rank"] = ranked_test["risk_score"].rank(method="first", ascending=False).astype(int)

ranked_test = ranked_test.sort_values("risk_score", ascending=False)

display(ranked_test.head(50))

# Save outputs for reporting.
ranked_test.to_csv("ranked_test_output.csv", index=False)
ranked_test.head(50).to_csv("top_50_ranked_test_output.csv", index=False)

print("Saved ranked_test_output.csv")
print("Saved top_50_ranked_test_output.csv")

In [ ]:
ranked_test

## 17. How to explain low recall@50

Use this explanation if your top-50 recall looks low.

Recall@50 is calculated as:

```text
true positives in top 50 / all true positives in the dataset
```

So if the top 50 contains 38 true positives, precision is high:

```text
precision@50 = 38 / 50 = 0.76
```

But recall may still look low if there are many positives overall. For example, if there are 200 positives:

```text
recall@50 = 38 / 200 = 0.19
```

That does not mean the model is useless. It means top 50 is too small to capture most positives.

That is why this notebook also reports:

```text
pct_of_max_possible_recall
```

This shows how close the model is to the best possible result for that value of K.

## 18. Interpretation prompts

Answer these in markdown after running the notebook.

1. What is the positive class and why is the problem imbalanced?
2. Why is accuracy not enough for this task?
3. What was the validation ROC-AUC?
4. What was the validation Average Precision?
5. In the validation top 50, how many true positives were captured?
6. What was precision@50?
7. What was recall@50?
8. What percentage of the maximum possible recall@50 did the model achieve?
9. Did tuning improve the baseline Random Forest?
10. Based on the test results, should the organisation use top 50, top 100, or top 200?
11. What extra data might improve the ranking?
12. What are the limitations of using this model operationally?

Suggested conclusion structure:

> The model should be used as a prioritisation tool, not as an automatic decision-maker. The most useful metric is whether the top-ranked records contain more true incidents than random selection. The final top-K results suggest that [...]. However, the model is limited by [...], so future work should [...].

## Plot the validation cumulative gains curve This plot shows how quickly the model captures true positives as more records are inspected.
## A useful model should rise faster than the diagonal/random line.

In [ ]:
gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)

ax = gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — tuned Random Forest",
)
ax.set_xlabel("Fraction of records inspected")
ax.set_ylabel("Fraction of positives captured")
plt.show()


15. Final test evaluation
Only run this after you have finished choosing the model using the validation set.

Do not tune the model again after looking at the test results.

In [ ]:
final_model = best_rf

test_scores = get_scores(final_model, X_test)

print("Final tuned Random Forest — test")
print(f"ROC-AUC:           {roc_auc_score(y_test, test_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_test, test_scores):.3f}")

test_top_k = top_k_capture_table(y_test, test_scores, TOP_K_VALUES, label="test")
display(test_top_k)

16. Create a ranked test output
This creates a table sorted from highest predicted risk to lowest predicted risk.

The top rows are the records the organisation would prioritise for inspection or intervention.

In [ ]:
ids_test

In [ ]:
ranked_test = ids_test.copy()
ranked_test["true_target"] = y_test.values
ranked_test["risk_score"] = test_scores
ranked_test["risk_rank"] = ranked_test["risk_score"].rank(method="first", ascending=False).astype(int)

ranked_test = ranked_test.sort_values("risk_score", ascending=False)

display(ranked_test)#.head(50))

# Save outputs for reporting.
ranked_test.to_csv("ranked_test_output.csv", index=False)
#ranked_test.head(50).to_csv("top_50_ranked_test_output.csv", index=False)

#print("Saved ranked_test_output.csv")
#print("Saved top_50_ranked_test_output.csv")

17. How to explain low recall@50
Use this explanation if your top-50 recall looks low.

Recall@50 is calculated as:

true positives in top 50 / all true positives in the dataset
So if the top 50 contains 38 true positives, precision is high:

precision@50 = 38 / 50 = 0.76
But recall may still look low if there are many positives overall. For example, if there are 200 positives:

recall@50 = 38 / 200 = 0.19
That does not mean the model is useless. It means top 50 is too small to capture most positives.

That is why this notebook also reports:

pct_of_max_possible_recall
This shows how close the model is to the best possible result for that value of K.

18. Interpretation prompts
Answer these in markdown after running the notebook.

What is the positive class and why is the problem imbalanced?
Why is accuracy not enough for this task?
What was the validation ROC-AUC?
What was the validation Average Precision?
In the validation top 50, how many true positives were captured?
What was precision@50?
What was recall@50?
What percentage of the maximum possible recall@50 did the model achieve?
Did tuning improve the baseline Random Forest?
Based on the test results, should the organisation use top 50, top 100, or top 200?
What extra data might improve the ranking?
What are the limitations of using this model operationally?
Suggested conclusion structure:

The model should be used as a prioritisation tool, not as an automatic decision-maker. The most useful metric is whether the top-ranked records contain more true incidents than random selection. The final top-K results suggest that [...]. However, the model is limited by [...], so future work should [...].




In [ ]:
### Extra part.
#### Join geolocation result:
Gazetter = pd.read_excel('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/OXFORDSHIRE_GAZETTER.xlsx')

In [ ]:
Gazetter = pd.DataFrame(Gazetter)
Gazetter.head(2)
Gazetter.drop(columns = ['BUILDINGNAME', 'BUILDINGNUMBER', 'STREETNAME1',
       'STREETNAME2', 'AREANAME1', 'AREANAME2', 'POSTCODE', 'POSTTOWN',
       'POSTCOUNTY'], inplace = True)
ranked_geolocs = Gazetter.merge(ranked_test, left_on = 'FCL_URN', right_on = 'Addressbase UPRN', how = 'inner')
ranked_geolocs

In [ ]:
ranked_geolocs.to_csv('H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/ranked_geolocs_for_map.csv')